In [ ]:
import os 
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 


In [ ]:
import logging

log_file = Path(os.getcwd()) / "src/log_analysis.log"
logging.basicConfig(filename=log_file, level=logging.DEBUG)

In [ ]:
from src.class_imbalance_ratios.analysis.metric_run_dir_analysis import analyse_acc, analyse_loss, analyse_gc
from src.class_imbalance_ratios.analysis.needed_paths import outputs_dir

In [ ]:
display("=========== DISPLAYING GRADIENT CORRELATION METRICS ===========")
analyse_gc(outputs_dir=outputs_dir)

In [ ]:
display("=========== DISPLAYING LOSS METRICS ===========")
analyse_loss(outputs_dir=outputs_dir)

In [ ]:
display("=========== DISPLAYING ACCURACY METRICS ===========")
analyse_acc(outputs_dir=outputs_dir)

In [ ]:
"""
Standard metric analysis helpers. 
"""
from src.class_imbalance_ratios.analysis.summarise_metrics import MetricMetadata


def summarize_run_acc(run_metrics_paths: list[Path], prefix:str):
    """ 
    This summarises accuracy metrics across a run.
    Output is in the parent directory of the current path.  
    """
    enumerated_metrics = enumerate(run_metrics_paths)

    run_dfs:list[pd.DataFrame] = []
    for idx, train_metrics_path in enumerated_metrics:
        metric = torch.load(train_metrics_path.resolve(), map_location="cpu")
        classes = list(metric.keys())
        base_acc = MyMultiClassAccuracy(classes)
        base_acc.load_state_dict(metric)
        print(type(base_acc))

        # Convert tensors not addressed by `.compute()`
        clean_epochs = [tensor.item() for tensor in base_acc.metric_state["epochs"]]
        clean_steps = [tensor.item() for tensor in base_acc.metric_state["steps"]]

        metadata = MetricMetadata(epochs=clean_epochs, steps=clean_steps) # Unify values in new dict
        metadata_df = pd.DataFrame(metadata)
        head = metadata_df.head(10)
        index = pd.MultiIndex.from_frame(metadata_df)

        computed = base_acc.compute()

        data_df = pd.DataFrame(computed, index=index).replace(100, np.nan)
        run_dfs.append(data_df)

    df_all = pd.concat(run_dfs)
    mean_df = df_all.groupby(level=0).mean(skipna=True)
    std_df = df_all.groupby(level=0).std(skipna=True)
    summary_mean = mean_df.describe()
    summary_std = std_df.describe() 
    display(summary_mean)
    display(summary_std)
    viz_all_col(mean_df, unified_name="Train Accuracy Mean")    
    viz_all_col(std_df, unified_name="Train Accuracy Std")
    plt.show()
    print("shown")    